# 1. Create a Track

In **Rolland**, building a track model follows a strict object-oriented workflow. First, the individual physical components of the track (such as the rail, pads, sleepers, and ballast) must be created as independent instances. After these components are defined with their respective geometric, elastic, and damping properties, they are passed into a track class to be assembled into a complete `Track` model.

It is important to note that Rolland models a **single-sided track**. This means we simulate one rail and the corresponding half of the sleeper and track bed. Keep this assumption in mind when defining component masses and stiffnesses.

In this section, we will walk through this process step-by-step.

## 1.1 Defining a Rail

The fundamental component of the track is the rail. We can import standard pre-defined rail geometries directly from the database. Here, we load the standard `60E1` (formerly UIC60) rail profile.

In [1]:
from rolland.database.rail.db_rail import rail_60E1, available_rails

# Print all available rail profiles in the database
print("Available rail profiles:", available_rails())

# Load the standard 60E1 rail profile
rail = rail_60E1

Available rail profiles: ['rail_49E1', 'rail_49E5', 'rail_54E1', 'rail_54E3', 'rail_54E4', 'rail_60E1', 'rail_60E2']


## 1.2 Defining a Sleeper and Slab

Sleepers and slabs support the rails. To define a sleeper, we need its physical dimensions (length, width, height) and material density (e.g., concrete). 

**Assumptions & Calculations:**
*   The sleeper is assumed to be a uniform rectangular cuboid.
*   The mass (`m`) is computed as `density * volume`.
*   The moments of inertia (`I_sz`, `I_sy`, `I_sx`) are calculated using standard formulas for a rectangular block, and are normalized by the density as required by the model parameters.
*   To convert discrete sleeper properties into a continuous `Slab` model, the mass and inertia values are "smeared" by dividing them by the sleeper distance (`slep_dist`).
*   The top (`z_st`) and bottom (`z_sb`) coordinates are set relative to the center of mass (half the height in each direction).

In [2]:
from rolland import Sleeper, Slab

slep_dist = 0.6  # Distance between sleepers in meters

# Sleeper geometry and material
sl_len = 2.5       # Length (m)
sl_width = 0.245   # Width (m)
sl_height = 0.185  # Height (m)

rhos = 2648  # Density of concrete (kg/m^3)
m = rhos * sl_len * sl_width * sl_height  # Total sleeper mass (kg)

# Moments of inertia (normalized by density for the model input)
I_sz = (sl_len**2 + sl_width**2) * m / 12 * (1 / rhos)
I_sy = (sl_height**2 + sl_width**2) * m / 12 * (1 / rhos)
I_sx = (sl_len**2 + sl_height**2) * m / 12 * (1 / rhos)

# Define discrete Sleeper
sleeper = Sleeper(
    ms=m,
    rhos=rhos,
    Is_x=I_sx,
    Is_y=I_sy,
    Is_z=I_sz,
    lengs=sl_len,
    wdths=sl_width,
    heights=sl_height,
    z_st=-sl_height / 2,
    z_sb=sl_height / 2,
)

# Define equivalent continuous Slab
slab = Slab(
    ms=m / slep_dist,
    Is_x=I_sx / slep_dist,
    Is_y=I_sy / slep_dist,
    Is_z=I_sz / slep_dist,
    lengs=sl_len,
    rhos=rhos,
    equ_wdths=sl_width,
    heights=sl_height,
    z_st=-sl_height / 2,
    z_sb=sl_height / 2,
)

## 1.3 Defining Pads and Ballast

The track bed incorporates elastic and damping components—specifically the rail pads (between the rail and sleeper) and the ballast (beneath the sleeper). 

**Assumptions:**
*   Properties like stiffness (`sp`, `sb`) and loss factors (`etap`, `etab`) are defined for vertical (`z`), lateral (`y`), and longitudinal (`x`) degrees of freedom.
*   Just like with the sleeper, continuous equivalents (`ContPad` and continuous `Ballast`) are created by dividing the discrete stiffness properties by the sleeper distance (`slep_dist`).

```{figure} ../images/mech_props_pad_ballast.png
---
width: 100%
name: mech-props
---
Elastic and damping properties corresponding to the rail pad and the ballast for each degree of freedom (warping is not shown).
```

In [3]:
from rolland import DiscrPad, ContPad, Ballast

# Discrete Pad
discrpad = DiscrPad(
    sp_z=120e6,
    sp_y=40e6,
    sp_x=40e6,
    etap_z=0.2,
    etap_y=0.2,
    etap_x=0.2,
    etap_r=0.2,
    wdthp=0.15
)

# Continuous Pad
contpad = ContPad(
    sp_z=120e6 / slep_dist,
    sp_y=40e6 / slep_dist,
    sp_x=40e6 / slep_dist,
    etap_z=0.2,
    etap_y=0.2,
    etap_x=0.2,
    etap_r=0.2,
    wdthp=0.15
)

# Discrete Ballast
discrballast = Ballast(
    sb_z=120e6,
    sb_y=120e6,
    sb_x=120e6,
    etab_z=1,
    etab_y=2,
    etab_x=2,
    etab_r=2,
)

# Continuous Ballast
contballast = Ballast(
    sb_z=120e6 / slep_dist,
    sb_y=120e6 / slep_dist,
    sb_x=120e6 / slep_dist,
    etab_z=1,
    etab_y=2,
    etab_x=2,
    etab_r=2,
)

## 1.4 Assemble the Track and Domain

Now that all components (both discrete and continuous) are fully defined, we can pass them into a Track class (e.g., `SimplePeriodicBallastedSingleRailTrack` or `ContSlabSingleRailTrack`) and set up the computational domain with PML boundaries to simulate infinite track extensions.

In [4]:
from rolland.track import SimplePeriodicBallastedSingleRailTrack
from rolland.domainsetup import DomSetup
from rolland.boundary import CFSPML

# Assemble a discrete ballasted track
track = SimplePeriodicBallastedSingleRailTrack(
    rail=rail,
    pad=discrpad,
    sleeper=sleeper,
    ballast=discrballast,
    distance=slep_dist, z_f=0.081, y_f=0.0,
)

# Setup the domain
domain = DomSetup(
    track=track,
    bound=CFSPML(l_bound=15.0)
)

print(f"Track assembled! Grid points nx: {domain.nx}")


Track assembled! Grid points nx: 1188
